# SQL

In [59]:
import sqlite3
import os

In [60]:
# 1. Connect
connection = sqlite3.connect("sql.db") 

In [61]:
# 2 CREATE TABLE
table_creation_query = """ 
CREATE TABLE IF NOT EXISTS employe (
    emp_id INTEGER PRIMARY KEY,
    first_name TEXT NOT NULL,
    last_name TEXT NOT NULL,
    email TEXT UNIQUE NOT NULL,
    hire_date TEXT NOT NULL,
    salary REAL NOT NULL
);
"""
table_creation_query2 = """ 
CREATE TABLE IF NOT EXISTS customers (
    customer_id INTEGER PRIMARY KEY AUTOINCREMENT,
    first_name TEXT NOT NULL,
    last_name TEXT NOT NULL,
    email TEXT UNIQUE NOT NULL,
    phone TEXT
);
"""

In [62]:
cursor = connection.cursor() 

In [63]:
cursor.execute(table_creation_query)
cursor.execute(table_creation_query2)

In [64]:
# 3. Insert queries
insert_query_employe = """
INSERT OR IGNORE INTO employe (emp_id, first_name, last_name, email, hire_date, salary)
VALUES (?, ?, ?, ?, ?, ?)
"""

insert_query_customers = """ 
INSERT OR IGNORE INTO customers (customer_id, first_name, last_name, email, phone)
VALUES (?, ?, ?, ?, ?)
"""

In [65]:
# 4. Sample data
employe_data = [
    (101, "Samantha", "Shah", "samantha.shah@example.com", "2025-12-22", 75000.00),
    (102, "Amit", "Verma", "amit.verma@example.com", "2024-06-15", 68000.00),
    (103, "Riya", "Patel", "riya.patel@example.com", "2023-03-10", 72000.00),
    (104, "Karan", "Singh", "karan.singh@example.com", "2022-11-05", 80000.00),
    (105, "Neha", "Gupta", "neha.gupta@example.com", "2025-01-20", 70000.00)
]

customers_data = [
    (201, "Rahul", "Mehta", "rahul.mehta@example.com", "9876543210"),
    (202, "Priya", "Kumar", "priya.kumar@example.com", "9123456780"),
    (203, "Ankit", "Sharma", "ankit.sharma@example.com", "9988776655"),
    (204, "Sneha", "Joshi", "sneha.joshi@example.com", "9012345678"),
    (205, "Vikram", "Rao", "vikram.rao@example.com", "9876123450")
]

In [66]:
cursor.executemany(insert_query_employe,employe_data)
cursor.executemany(insert_query_customers,customers_data)

In [67]:
connection.commit()

In [68]:
cursor.execute("select * from employe;")
cursor.fetchall()

[(101, 'Samantha', 'Shah', 'samantha.shah@example.com', '2025-12-22', 75000.0),
 (102, 'Amit', 'Verma', 'amit.verma@example.com', '2024-06-15', 68000.0),
 (103, 'Riya', 'Patel', 'riya.patel@example.com', '2023-03-10', 72000.0),
 (104, 'Karan', 'Singh', 'karan.singh@example.com', '2022-11-05', 80000.0),
 (105, 'Neha', 'Gupta', 'neha.gupta@example.com', '2025-01-20', 70000.0)]

# TOOL

In [69]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq

In [70]:
api_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(api_key=api_key,model="llama-3.1-8b-instant")

In [71]:
from langchain_community.utilities import SQLDatabase
db = SQLDatabase.from_uri("sqlite:///sql.db")

In [72]:
print("Dialect:",db.dialect)
print("Usable tables:", db.get_usable_table_names())

Dialect: sqlite
Usable tables: ['customers', 'employe']


In [73]:
query = db.run("select first_name,last_name,salary from employe where salary > 70000")
print("ANSWER:- ",query)

ANSWER:-  [('Samantha', 'Shah', 75000.0), ('Riya', 'Patel', 72000.0), ('Karan', 'Singh', 80000.0)]


In [74]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit
toolkit = SQLDatabaseToolkit(db=db,llm=llm)

In [75]:
tools = toolkit.get_tools()

In [76]:
tools

[QuerySQLDatabaseTool(description="Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.", db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x0000019EBE8776D0>),
 InfoSQLDatabaseTool(description='Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3', db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x0000019EBE8776D0>),
 ListSQLDatabaseTool(db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x0000019EBE8776D0>),
 QuerySQLCheckerTool(description='Use this tool to 

In [77]:
for tool in tools:
    print(tool.name)

sql_db_query
sql_db_schema
sql_db_list_tables
sql_db_query_checker


In [78]:
list_tables_tool= next((tool for tool in tools if tool.name == "sql_db_list_tables"),None)
list_tables_tool

ListSQLDatabaseTool(db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x0000019EBE8776D0>)

In [79]:
get_schema_tool = next((tool for tool in tools if tool.name == "sql_db_schema"),None)
get_schema_tool

InfoSQLDatabaseTool(description='Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3', db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x0000019EBE8776D0>)

In [80]:
print("List Of Table:- ",list_tables_tool.invoke(""))
print("*" * 80)
print("Under the {customers or employe}:- ",get_schema_tool.invoke("customers"))

List Of Table:-  customers, employe
********************************************************************************
Under the {customers or employe}:-  
CREATE TABLE customers (
	customer_id INTEGER, 
	first_name TEXT NOT NULL, 
	last_name TEXT NOT NULL, 
	email TEXT NOT NULL, 
	phone TEXT, 
	PRIMARY KEY (customer_id), 
	UNIQUE (email)
)

/*
3 rows from customers table:
customer_id	first_name	last_name	email	phone
201	Rahul	Mehta	rahul.mehta@example.com	9876543210
202	Priya	Kumar	priya.kumar@example.com	9123456780
203	Ankit	Sharma	ankit.sharma@example.com	9988776655
*/


In [87]:
#  sql function
def db_query_tool(query: str) -> str:
    """
    execute a SQL query against the databese and result the result.
    if the query is invalid or returns no result, an error message will be returened.
    in case of an error,the uer is advised to rewrite the query and try again
    """
    result = db.run_no_throw(query)

In [84]:
print(type(db_query_tool))

<class 'function'>


In [85]:
print(db_query_tool("SELECT * FROM employe limit 10;"))

None


In [ ]:
tools = []

In [ ]:
# just explain it :- db.run_no_throw(query)
class Database:
    def run_no_throw(self,query):
        try:
            # Assume 'self.connection' is a valid database connection
            cursor = self.connection.cursor()
            cursor.execute(query)
            return cursor.fetchall()    # Or anather method to retrieve results
        except Exception as e:
            print(f"Error executing query: {e}")
            return None     # Fallback value

In [ ]:
from typing import TypedDict,Annotated
from langgraph.graph.message import add_messages,AnyMessage

class sql(TypedDict):
    messages : Annotated[list[AnyMessage], add_messages]

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = """ 
You are a SQL expert with a strong attention to detail.
Double check the SQLLite query for comman mistakes,including:
- Using NOT IN with NULL values.
- Using UNION whene UNIONE ALL should have been used.
- Using BETWEEN for execlution ranges
- DATA type mismatch in prediction
- Properly quoting identifiers
- Using the correct number of arguments for function
- Casting to the correct data type 
- Using the proper columns for joins

if there are any of the above mistakes, rewrite the query. if there are no mistkes, just reproduce the original query
you will call the appropriate tool to execute the query after running this check
"""

query_check_promp = ChatPromptTemplate.from_messages(
    [('system',prompt),('placeholder', "{placeholder}" )]
)

query_check = prompt | llm.bind_tools([[db_query_tool]])

query_check.invoke()
